In [1]:
#Get the SQLite database to use
import sqlite3

In [2]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [3]:
# Create Students table
cursor.execute("""
    CREATE TABLE Students (
        student_id INTEGER PRIMARY KEY,
        name TEXT
    )
""")

In [4]:
# Create Subjects table
cursor.execute("""
    CREATE TABLE Subjects (
        subject_id INTEGER PRIMARY KEY,
        name TEXT
    )
""")

In [5]:
# Create Grades table (links students to subjects)
cursor.execute("""
    CREATE TABLE Grades (
        student_id INTEGER,
        subject_id INTEGER,
        grade TEXT,
        FOREIGN KEY(student_id) REFERENCES Students(student_id),
        FOREIGN KEY(subject_id) REFERENCES Subjects(subject_id)
    )
""")

In [6]:
# Insert sample data
students = [(1, "Alice"), (2, "Bob"), (3, "Charlie")]
subjects = [(101, "Math"), (102, "Science"), (103, "History")]
grades = [(1, 101, "A"), (1, 102, "B"), (2, 101, "C"), (3, 103, "A")]

In [7]:
cursor.executemany("INSERT INTO Students VALUES (?, ?)", students)
cursor.executemany("INSERT INTO Subjects VALUES (?, ?)", subjects)
cursor.executemany("INSERT INTO Grades VALUES (?, ?, ?)", grades)

In [9]:
# To select all data from the "Students" table and print

cursor.execute("SELECT * FROM students")
data = cursor.fetchall()

# Print the data
for row in data:
    print(row)

(1, 'Alice')
(2, 'Bob')
(3, 'Charlie')


In [10]:
# Perform a join to get students' grades for subjects they registered for
cursor.execute("""
    SELECT Students.name, Subjects.name, Grades.grade
    FROM Grades
    JOIN Students ON Grades.student_id = Students.student_id
    JOIN Subjects ON Grades.subject_id = Subjects.subject_id
""")

# Fetch and display the results
results = cursor.fetchall()
for row in results:
    print(row)

# Close the connection
conn.close()

('Alice', 'Math', 'A')
('Alice', 'Science', 'B')
('Bob', 'Math', 'C')
('Charlie', 'History', 'A')


Try converting the data into portable format of JSON as NoSQL version of data

In [12]:
import json
import pandas as pd

In [13]:
# Convert results to JSON format
data_json = [{"student": row[0], "subject": row[1], "grade": row[2]} for row in results]
json_filename = "grades.json"
with open(json_filename, "w") as json_file:
    json.dump(data_json, json_file, indent=4)

In [16]:
# Load the JSON data from the file
with open("grades.json", "r") as json_file:
    grades_data = json.load(json_file)

# Now you can work with the JSON data loaded into the `grades_data` variable
print(grades_data)

[{'student': 'Alice', 'subject': 'Math', 'grade': 'A'}, {'student': 'Alice', 'subject': 'Science', 'grade': 'B'}, {'student': 'Bob', 'subject': 'Math', 'grade': 'C'}, {'student': 'Charlie', 'subject': 'History', 'grade': 'A'}]


In [22]:
# Print the JSON data with indentation
print(json.dumps(grades_data, indent=1))

[
 {
  "student": "Alice",
  "subject": "Math",
  "grade": "A"
 },
 {
  "student": "Alice",
  "subject": "Science",
  "grade": "B"
 },
 {
  "student": "Bob",
  "subject": "Math",
  "grade": "C"
 },
 {
  "student": "Charlie",
  "subject": "History",
  "grade": "A"
 }
]


In [23]:
# Convert results to an Excel file
df = pd.DataFrame(data_json)
print(df)

   student  subject grade
0    Alice     Math     A
1    Alice  Science     B
2      Bob     Math     C
3  Charlie  History     A


In [25]:
excel_filename = "grades.xlsx"
df.to_excel(excel_filename, index=False)

# Close the connection
conn.close()

# Return the file paths
json_filename, excel_filename

('grades.json', 'grades.xlsx')